# Discussion Chapter Draft

Converted from the original Python workflow script so the dissertation repository uses notebook-based workflow artefacts.


In [ ]:
from pathlib import Path

from docx import Document
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor


ROOT = Path("/Users/lu_nanxi/CASA/Dissertation_Data")
OUT = ROOT / "dissertation_discussion_chapter_draft_vivacity.docx"


def set_cell_shading(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)


def set_cell_margins(cell, top=80, start=110, bottom=80, end=110):
    tc = cell._tc
    tc_pr = tc.get_or_add_tcPr()
    tc_mar = tc_pr.first_child_found_in("w:tcMar")
    if tc_mar is None:
        tc_mar = OxmlElement("w:tcMar")
        tc_pr.append(tc_mar)
    for m, v in [("top", top), ("start", start), ("bottom", bottom), ("end", end)]:
        node = tc_mar.find(qn(f"w:{m}"))
        if node is None:
            node = OxmlElement(f"w:{m}")
            tc_mar.append(node)
        node.set(qn("w:w"), str(v))
        node.set(qn("w:type"), "dxa")


def set_table_borders(table):
    tbl_pr = table._tbl.tblPr
    borders = tbl_pr.first_child_found_in("w:tblBorders")
    if borders is None:
        borders = OxmlElement("w:tblBorders")
        tbl_pr.append(borders)
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        elem = borders.find(qn(f"w:{edge}"))
        if elem is None:
            elem = OxmlElement(f"w:{edge}")
            borders.append(elem)
        elem.set(qn("w:val"), "single")
        elem.set(qn("w:sz"), "4")
        elem.set(qn("w:space"), "0")
        elem.set(qn("w:color"), "DADCE0")


def add_run(paragraph, text, bold=False, italic=False, size=11, color="000000"):
    run = paragraph.add_run(text)
    run.bold = bold
    run.italic = italic
    run.font.name = "Arial"
    run._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    run.font.size = Pt(size)
    run.font.color.rgb = RGBColor.from_string(color)
    return run


def add_para(doc, text):
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(8)
    p.paragraph_format.line_spacing = 1.15
    add_run(p, text)
    return p


def add_heading(doc, text, level=1):
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(18 if level == 1 else 13)
    p.paragraph_format.space_after = Pt(6)
    add_run(p, text, size=19 if level == 1 else 14)
    return p


def add_bullets(doc, items):
    for item in items:
        p = doc.add_paragraph(style="List Bullet")
        p.paragraph_format.space_after = Pt(4)
        p.paragraph_format.line_spacing = 1.15
        add_run(p, item)


def add_claim_table(doc):
    table = doc.add_table(rows=1, cols=3)
    table.autofit = False
    for i, width in enumerate([1.65, 2.55, 2.2]):
        table.columns[i].width = Inches(width)
    set_table_borders(table)
    headers = ["Claim type", "Defensible wording", "Avoid wording"]
    for i, header in enumerate(headers):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        add_run(cell.paragraphs[0], header, bold=True, size=9)

    rows = [
        (
            "Scheme effect",
            "The models show mixed and mostly uncertain treated-control trajectory differences.",
            "The schemes did not work, or caused reductions in active travel.",
        ),
        (
            "Equity",
            "The usable causal sample is weighted toward deprived contexts, with limited affluent comparison evidence.",
            "Deprived and affluent areas respond differently to interventions.",
        ),
        (
            "Method",
            "The dissertation demonstrates a reproducible data-quality and matched-control workflow under real monitoring constraints.",
            "The model produces definitive causal impact estimates.",
        ),
        (
            "Policy",
            "Confirmed exact scheme dates and longer pre-intervention monitoring strengthen local evaluation.",
            "Sensor data alone can fully explain active travel uptake.",
        ),
    ]
    for row in rows:
        cells = table.add_row().cells
        for i, value in enumerate(row):
            set_cell_margins(cells[i])
            cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            p = cells[i].paragraphs[0]
            p.paragraph_format.space_after = Pt(0)
            if i == 0:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            add_run(p, value, size=8.6)


def add_callout(doc, title, text):
    table = doc.add_table(rows=1, cols=1)
    set_table_borders(table)
    cell = table.rows[0].cells[0]
    set_cell_shading(cell, "F8F9FA")
    set_cell_margins(cell, top=120, start=140, bottom=120, end=140)
    p = cell.paragraphs[0]
    p.paragraph_format.space_after = Pt(0)
    add_run(p, title + ": ", bold=True, size=10)
    add_run(p, text, size=10)


def build_doc():
    doc = Document()
    section = doc.sections[0]
    section.top_margin = Inches(1)
    section.bottom_margin = Inches(1)
    section.left_margin = Inches(1)
    section.right_margin = Inches(1)

    styles = doc.styles
    styles["Normal"].font.name = "Arial"
    styles["Normal"]._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    styles["Normal"].font.size = Pt(11)

    title = doc.add_paragraph()
    title.paragraph_format.space_after = Pt(3)
    add_run(title, "Draft Discussion Chapter: Vivacity Active Travel Analysis", size=25)

    subtitle = doc.add_paragraph()
    subtitle.paragraph_format.space_after = Pt(12)
    add_run(
        subtitle,
        "Working prose connecting exploratory results to literature, equity, and dissertation claims",
        size=11,
        color="555555",
    )

    add_heading(doc, "5.1 Chapter Overview", 1)
    add_para(
        doc,
        "This chapter discusses what the Vivacity analysis contributes to understanding active travel interventions in the Liverpool City Region. The central empirical finding is cautious: the available treated-control models do not show a consistent post-intervention increase in walking or cycling relative to matched controls. Instead, the results are mixed, mostly statistically uncertain, and constrained by the use of first-day-of-installation-month intervention dates, sensor start dates, and the availability of valid controls.",
    )
    add_para(
        doc,
        "This does not make the analysis unsuccessful. Rather, it clarifies the real evaluation challenge: local active travel monitoring data can support transparent exploratory comparison, but only when sensor coverage, intervention timing, and control-site validity are strong enough. In this dissertation, those conditions are only partly met.",
    )

    add_heading(doc, "5.2 Interpreting the Absence of Clear Uptake", 1)
    add_para(
        doc,
        "The lack of a clear positive treated-control effect should not be interpreted simply as evidence that active travel schemes failed. Built-environment interventions may influence behaviour gradually, unevenly, or only when supported by wider network connectivity, safety perceptions, route continuity, and social acceptability. Aldred (2019) argues that active travel infrastructure should be understood within broader systems of mobility, rather than as isolated pieces of physical provision. This is consistent with the current findings: changes at individual countlines may not fully capture how a route is used, substituted, or avoided within the wider network.",
    )
    add_para(
        doc,
        "The results also align with cautions in the evaluation literature. Mölenberg et al. (2019) highlight that observational studies of cycling infrastructure often struggle to strengthen causal inference because intervention exposure, comparison groups, and pre-intervention trends are difficult to define. The current analysis faces related issues: confirmed exact scheme dates are now available, but some sensors may still begin producing reliable data after the confirmed date, and candidate control sites may still be exposed to other local transport changes.",
    )

    add_heading(doc, "5.3 Equity and Neighbourhood Context", 1)
    add_para(
        doc,
        "The dissertation question places equity at the centre by asking whether deprived and affluent neighbourhoods exhibit different active travel trajectories. The available data support a more modest answer than originally intended. The causal modelling set is weighted toward more deprived areas, with limited representation of more affluent LSOAs. This makes it possible to describe the deprivation context of the schemes and controls, but not to estimate a robust deprivation-stratified intervention effect.",
    )
    add_para(
        doc,
        "This limitation is substantively important. Equity in active travel cannot be assessed only by asking whether infrastructure exists in deprived areas. Aldred et al. (2021) show that distributional questions also depend on where new infrastructure is located, who benefits, and whether investment patterns reproduce or challenge existing inequalities. In this dissertation, the main contribution is to show that even when deprived areas are represented in the monitoring dataset, the evaluation design may still lack the balanced comparison structure needed to test differential uptake confidently.",
    )
    add_para(
        doc,
        "The findings should therefore be framed as an equity-context analysis rather than a definitive equity-impact analysis. The study can say that the usable evidence base is concentrated in deprived and middle-deprivation settings, and that this affects what can be inferred. It cannot yet say that deprived neighbourhoods respond more or less positively than affluent neighbourhoods after intervention.",
    )

    add_heading(doc, "5.4 Contribution of the Data Workflow", 1)
    add_para(
        doc,
        "A key contribution of the dissertation is methodological. Automated pedestrian and cycling counters offer high-frequency local evidence, but they are not analysis-ready. Lee and Sener (2020) note that emerging pedestrian and bicycle data sources expand monitoring possibilities while raising issues of coverage, validity, and interpretation. Similarly, recent work on counter validation emphasises that automated counts require careful quality assessment before being used as evidence of behavioural change.",
    )
    add_para(
        doc,
        "The integrity gate developed here responds to this challenge. It records when each countline first becomes reliable, removes incomplete or low-availability days, checks for duplicate daily keys, separates causal and descriptive-only uses, and documents why controls are included or excluded. This process is important because without it, apparent before/after changes could reflect sensor absence, missing data, or poor control selection rather than genuine changes in walking and cycling.",
    )

    add_heading(doc, "5.5 Relationship to Interrupted Time-Series Literature", 1)
    add_para(
        doc,
        "The modelling strategy draws on interrupted time-series logic, which is widely used when randomised evaluation is not possible. Lopez Bernal et al. (2016) emphasise the importance of modelling pre-existing trends, post-intervention level change, and post-intervention slope change. This dissertation follows that logic by estimating immediate treated-control differences and additional post-intervention trajectory differences, while also controlling for seasonality and countline fixed effects.",
    )
    add_para(
        doc,
        "However, the dissertation should be clear that this is not a textbook interrupted time-series evaluation. The intervention months are known, but the day-level intervention dates are represented by the first day of the provided installation month, and several schemes lack long pre-intervention treated data. The approach is therefore closer to an exploratory matched-control time-series design. Xiao et al. (2022) provide a useful comparator because their work uses interrupted time-series analysis to examine cycle infrastructure impacts, but the current dissertation has more constrained local monitoring data and therefore weaker causal leverage.",
    )

    add_heading(doc, "5.6 What Can Be Claimed", 1)
    add_claim_table(doc)
    add_callout(
        doc,
        "Recommended position",
        "The dissertation should present the modelling as transparent exploratory evidence, and the strongest contribution as a combined workflow for sensor cleaning, contextual joining, matched-control selection, and cautious interpretation under data uncertainty.",
    )

    add_heading(doc, "5.7 Policy and Monitoring Implications", 1)
    add_para(
        doc,
        "For local authorities and transport bodies, the main implication is that scheme monitoring needs to be planned before or alongside implementation. If sensors are installed after a scheme opens, or if only month-level installation timing is available rather than exact opening/completion dates, the ability to evaluate impact is weakened. A robust monitoring framework would record exact opening dates, retain metadata on route changes, identify plausible non-intervention controls in advance, and maintain consistent data access over multiple years.",
    )
    add_para(
        doc,
        "The analysis also suggests that equity evaluation requires more than attaching IMD deciles after the event. To evaluate whether active travel schemes produce different trajectories in deprived and affluent neighbourhoods, the monitoring design must include enough schemes and controls across the deprivation spectrum. Otherwise, deprivation becomes descriptive context rather than a variable that can be tested statistically.",
    )

    add_heading(doc, "5.8 Limitations", 1)
    add_bullets(
        doc,
        [
            "Confirmed scheme dates are encoded in the current analysis.",
            "Reliable pre-intervention treated data are unavailable for 12b and 12e, so these schemes cannot support the main treated-control model.",
            "The control pool remains limited and requires manual site checks to confirm absence of concurrent interventions.",
            "The number of usable schemes is too small for a robust deprivation-by-intervention interaction model.",
            "Countline data measure flows at specific points and cannot capture wider route substitution, trip purpose, or individual-level uptake.",
            "The Census and IMD variables describe residential LSOA context, not the socio-economic profile of people passing each sensor.",
        ],
    )

    add_heading(doc, "5.9 Future Work", 1)
    add_para(
        doc,
        "Future work should manually verify all control sites and test alternative matched controls. The analysis could then be rerun with sensitivity tests using alternative intervention dates, stricter seasonal controls, and separate pedestrian and cyclist models. If additional non-intervention sensors can be obtained, the matching process could be strengthened and the deprivation comparison made more balanced.",
    )
    add_para(
        doc,
        "A stronger equity evaluation would combine sensor counts with qualitative or survey evidence about perceived safety, route usefulness, and barriers to use among under-represented groups. This would connect observed flows with the social mechanisms that systematic reviews identify as important for active travel uptake, including access, confidence, safety, and everyday trip constraints.",
    )

    add_heading(doc, "5.10 Chapter Conclusion", 1)
    add_para(
        doc,
        "The dissertation's findings are best understood as a careful, critical evaluation of what current monitoring data can and cannot show. The models do not provide clear evidence that the selected schemes increased walking or cycling relative to controls, and they do not support a strong deprived-versus-affluent causal comparison. However, the analysis makes a useful contribution by exposing the data conditions required for credible active travel evaluation. It shows that equity-focused monitoring depends not only on where schemes are built, but also on whether data systems are capable of measuring change before and after intervention across comparable neighbourhood contexts.",
    )

    doc.save(OUT)


if __name__ == "__main__":
    build_doc()
    print(OUT)
